In [ ]:
import random
import requests
from io import StringIO
from Bio import SeqIO
from Bio.SeqRecord import SeqRecord


def fetch_proteins(size=50) -> list[SeqRecord]:
    params = {
        "query": "(reviewed:true) AND (length:[100 TO 300]) AND (organism_id:9606)",
        "format": "fasta",
        "size": size
    }

    response = requests.get("https://rest.uniprot.org/uniprotkb/search", params=params)

    if response.status_code != 200:
        raise Exception("Error while accessing UnitProt")

    fasta_data = response.text
    sequences = list(SeqIO.parse(StringIO(fasta_data), "fasta"))

    return sequences

def sample_pairs(sequences: list[SeqRecord], n_pairs=10) -> list[tuple[SeqRecord, SeqRecord]]:
    pairs = []
    for _ in range(n_pairs):
        sequence1, sequence2 = random.sample(sequences, 2)
        pairs.append((sequence1, sequence2))

    return pairs

In [ ]:
sequences = fetch_proteins(size=100)
pairs = sample_pairs(sequences, n_pairs=20)
print(f"{len(sequences)} downloaded sequences")
print(f"{len(pairs)} generated pairs")

In [ ]:
population_size: int = 200
max_iter: int = 500

In [ ]:
import time
from gwo import GreyWolfOptimizer


seq1, seq2 = pairs[random.randint(0, len(pairs))]
print(f"Alinhando as seguintes proteínas:", seq1.name, seq2.name, sep="\n")

optimizer = GreyWolfOptimizer(population_size, max_iter, str(seq1.seq), str(seq2.seq))

start_time = time.perf_counter()
convergence, best_fitness = optimizer.run()
end_time = time.perf_counter()

elapsed_time = end_time - start_time

optimizer.print_best_alignment()

print(f"Tempo de execução: {elapsed_time * 1000:.2f} ms")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(convergence, linewidth=2)
plt.xlabel("Iteração")
plt.ylabel("Melhor fitness score")
plt.title("Curva de convergência do GWO")
plt.grid(True, alpha=0.3)
plt.show()

print(f"\nAnálise de Convergência:")
print(f"Fitness inicial: {convergence[0]:.2f}")
print(f"Fitness final: {convergence[-1]:.2f}")
print(f"Melhoria: {convergence[-1] - convergence[0]:.2f}")
print(f"Melhor score encontrado: {best_fitness:.2f}")